In [ ]:
# Colab setup: install packages not preinstalled on Colab (safe to re-run)
!pip install -q abess

# Regularization for Model Uncertainty — Cross-Country Economic Growth (Comparative Political Economy)  ·  **Day 1 tutorial**

> **Source & framing.** This example uses the growth-determinants data of **Sala-i-Martin (1997), "I Just Ran
> Two Million Regressions," *American Economic Review* 87(2):178–183**, as compiled by **Fernández, Ley &
> Steel (2001), "Model Uncertainty in Cross-Country Growth Regressions," *Journal of Applied Econometrics*
> 16(5):563–576** — **72 countries, 41 candidate predictors** from competing theories. Note: these authors do
> **not** advocate one big OLS; their point is the opposite — with p/n ≈ 0.57 *no single kitchen-sink
> regression is trustworthy*, which is why they turn to model selection / Bayesian averaging. We use the same
> data to show how **regularization** gives a principled single-model answer to that model-uncertainty
> problem. (So, unlike the other tutorials, the OLS below is a **naive benchmark to be beaten**, not a
> published result we are reproducing.)

## Background

Why do some countries grow rich while others stagnate? Decades of growth economics produced **dozens of
competing theories** — initial income and convergence, human capital, investment, institutions and the rule of
law, openness to trade, geography, religion and colonial history — and therefore **dozens of candidate
predictors**, but only about **70 countries** with comparable long-run data. Running one big regression with
all of them **over-fits**; running many small ones invites cherry-picking. Sala-i-Martin (1997) famously "ran
two million regressions," and Fernández, Ley & Steel (2001) formalized the problem as **model uncertainty**.
Here we take the same 72-country, 41-predictor design and show how modern **regularization** confronts it.

## Data and codebook

**Unit of analysis:** a **country**; n = 72. Outcome `y` is average annual GDP per-capita growth. The 41
predictors are the Sala-i-Martin determinants; a representative selection:

| Variable | Definition |
|---|---|
| `y` | average GDP per-capita growth **(outcome)** |
| `GDP60` | GDP per capita in 1960 (tests **conditional convergence** — expected negative) |
| `LifeExp` | life expectancy (human capital / health) |
| `PrScEnroll` | primary-school enrollment (human capital) |
| `EquipInv` | equipment investment share |
| `NequipInv` | non-equipment investment share |
| `RuleofLaw` | rule-of-law index (institutions) |
| `YrsOpen` | years the economy was open to trade |
| `SubSahara`, `LatAmerica` | regional dummies |
| `Confucian`, `Muslim`, `Protestants`, `Catholic`, ... | religious-composition shares |
| `Mining`, `Area`, `Popg`, `WarDummy`, `PolRights`, ... | resource, size, demographic, conflict, political controls |

(The full 41-predictor list is loaded automatically below; see Sala-i-Martin 1997 for definitions.)

## Descriptive results

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.linear_model import LassoCV, RidgeCV, ElasticNetCV, LinearRegression, lasso_path
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
rng = np.random.RandomState(2026)

In [ ]:
d = pd.read_csv('https://raw.githubusercontent.com/desmarais-lab/desmarais-lab.github.io/master/istanbul_bilgi_ml_files/data/growth_determinants.csv')
outcome = 'y'
preds = [c for c in d.columns if c not in ('country', outcome)]
print(f'{d.shape[0]} countries x {len(preds)} predictors  (p/n = {len(preds)/d.shape[0]:.2f})')
d[[outcome]+preds].describe().T[['mean','std','min','max']].round(3).head(10)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
ax[0].hist(d[outcome], bins=15, color='#a6cee3', edgecolor='white')
ax[0].set_title('Outcome: GDP-per-capita growth'); ax[0].set_xlabel('avg. annual growth')
cors = d[preds].corrwith(d[outcome]).sort_values()
top = pd.concat([cors.head(6), cors.tail(6)])
ax[1].barh(top.index, top.values, color=['#1f78b4' if v>0 else '#e31a1c' for v in top.values])
ax[1].axvline(0, color='grey'); ax[1].set_title('Strongest correlations with growth'); plt.tight_layout()

## The naive benchmark: one regression with all 41 predictors

Putting **all 41 candidate predictors** into a single OLS is the "kitchen-sink" specification whose fragility
motivated Sala-i-Martin (1997) and Fernández, Ley & Steel (2001) in the first place. We fit it here **not as a
result to endorse but as a benchmark to beat** — to see concretely how it fails.

In [ ]:
X = d[preds].values.astype(float); y = d[outcome].values.astype(float)
ols = sm.OLS(y, sm.add_constant(X)).fit()
print(f'Full-model OLS: n = {int(ols.nobs)}, p = {len(preds)}, in-sample R^2 = {ols.rsquared:.3f} (adj. {ols.rsquared_adj:.3f})')
# famous standardized coefficients
Xs = StandardScaler().fit_transform(X)
bstd = pd.Series(sm.OLS(y, sm.add_constant(Xs)).fit().params[1:], index=preds)
print('\nKey standardized coefficients (growth-literature signs):')
for v in ['GDP60','LifeExp','PrScEnroll','EquipInv','RuleofLaw','YrsOpen','Confucian']:
    if v in bstd.index: print(f'   {v:12s} {bstd[v]:+.4f}')

**Reading the benchmark.** The full model fits the 72 countries almost perfectly in-sample
($R^2 \approx 0.96$) and recovers the canonical growth signs — **`GDP60` negative** (the celebrated
**conditional convergence** result: poorer 1960 countries grew faster) with **life expectancy, primary
schooling, equipment investment, and the rule of law positive**. But 41 predictors on 72 countries is a
textbook over-fit: the near-perfect in-sample fit is an illusion, and — as we'll see — the model predicts new
countries **disastrously**. This is exactly the fragility that led Sala-i-Martin and Fernández–Ley–Steel to
reject the single kitchen-sink regression in favor of model selection/averaging.

## Regularization & variable selection

With **41 predictors and 72 countries** (p/n $\approx$ 0.57), ordinary least squares has almost as many
parameters as observations and **massively over-fits**. Penalized regression — lasso ($L_1$), ridge ($L_2$),
elastic net — shrinks the coefficients to tame the variance, and the lasso selects a compact subset of robust
growth determinants.

In [ ]:
liCV = LassoCV(cv=5, random_state=0, max_iter=100000).fit(Xs, y)
n_keep = int(np.sum(liCV.coef_ != 0))
print(f'At the CV-optimal penalty the lasso keeps {n_keep} of {len(preds)} predictors (rest set to exactly 0).')
kept = pd.Series(liCV.coef_, index=preds)
kept = kept[kept != 0].reindex(kept[kept!=0].abs().sort_values(ascending=False).index)
print('\nLasso-selected determinants (by |coefficient|):')
print(kept.round(4).head(12).to_string())

**Which fits best out of sample?** We **repeat a 70/30 split 50 times**: each replicate tunes the penalty
by cross-validation on the training countries and scores once on the held-out countries, reporting held-out
**RMSE** and **$R^2$**.

In [ ]:
def rmse(a,b): return float(np.sqrt(mean_squared_error(a,b)))
REPS = 50; res = {k:[] for k in ['OLS','Lasso','Ridge','ElasticNet']}; r2 = {k:[] for k in res}
for r in range(REPS):
    Xtr,Xte,ytr,yte = train_test_split(X, y, test_size=0.30, random_state=2025+r)
    sc = StandardScaler().fit(Xtr); Ztr, Zte = sc.transform(Xtr), sc.transform(Xte)
    for name, mdl in [('OLS', LinearRegression()),
                      ('Lasso', LassoCV(cv=5, random_state=0, max_iter=100000)),
                      ('Ridge', RidgeCV(alphas=np.logspace(-2,4,50))),
                      ('ElasticNet', ElasticNetCV(cv=5, l1_ratio=0.5, random_state=0, max_iter=100000))]:
        p = mdl.fit(Ztr, ytr).predict(Zte); res[name].append(rmse(yte,p)); r2[name].append(r2_score(yte,p))
for k in res: print(f'{k:12s} held-out RMSE = {np.mean(res[k]):.4f}   held-out R^2 = {np.mean(r2[k]):+.2f}')
best = min(['Lasso','Ridge','ElasticNet'], key=lambda k: np.mean(res[k]))
print(f'\nBest penalization: {best}. {100*(np.mean(res["OLS"])-np.mean(res[best]))/np.mean(res["OLS"]):.0f}% lower held-out RMSE than OLS.')

This is the sharpest regularization lesson in the whole course: unpenalized OLS **over-fits so badly that
its held-out $R^2$ is enormously negative** — its predictions for new countries are far worse than simply
guessing the mean growth rate — while the penalized models achieve a **positive** held-out $R^2$ and cut RMSE
by more than half. With p/n near 1, a penalty is not a refinement, it is the difference between a model that
predicts and one that is useless out of sample.

In [ ]:
alphas, cpath, _ = lasso_path(Xs, y, n_alphas=50)
plt.figure(figsize=(6,3.4)); plt.plot(np.log10(alphas), cpath.T, color='#1f78b4', alpha=.35)
plt.xlabel('log10(alpha)  (more penalty ->)'); plt.ylabel('coefficient'); plt.title('Lasso coefficient paths (41 growth predictors)'); plt.tight_layout()

## Best-subset selection with ABESS

Lasso reaches a sparse model through shrinkage. **Best-subset selection** searches directly for the subset of
predictors that fits best; the **adaptive best-subset (ABESS)** algorithm does so efficiently even with 41
candidates and chooses the subset size automatically.

In [ ]:
try:
    from abess.linear import LinearRegression as AbessLR
    ab = AbessLR(support_size=range(1, 12)).fit(Xs, y)
    sel = [p for p,c in zip(preds, ab.coef_) if c!=0]
    print(f'ABESS selects a best subset of {len(sel)} predictors:')
    print('  ', sel)
except Exception as e:
    print('Install abess to run this cell:  !pip install abess')
    print('(', e, ')')

Best subset and the lasso converge on the same story the robust-growth literature tells — **conditional
convergence (`GDP60`), human capital, investment, and institutions** — recovered here not by running two
million regressions but by a single principled penalty.

## Takeaway

This example is regularization in its **natural habitat**: a real, famous cross-country growth regression with
almost as many predictors (41) as countries (72). Ordinary least squares fits the sample nearly perfectly yet
predicts new countries disastrously; lasso/ridge/elastic-net (`scikit-learn`) and best-subset selection
(`abess`) turn that useless over-fit into a model that **actually predicts out of sample** and **names the
compact set of robust growth determinants** — the modern answer to the model-uncertainty problem Sala-i-Martin
and Fernández–Ley–Steel posed.

## Recommended exercises

1. Compare the lasso's selected determinants to Sala-i-Martin's (1997) "robust" list — how much do they agree?
2. Vary the elastic-net `l1_ratio` from 0 (ridge) to 1 (lasso); how do held-out RMSE and the number kept trade off?
3. Standardize vs not: refit without scaling and watch how the penalty mis-behaves across differently-scaled predictors.
4. Drop the regional/religion dummies and re-run; how much of the predictive signal survives?